In [1]:
   import os
   from dotenv import load_dotenv
   from sqlalchemy import create_engine, text

   load_dotenv()
   url = (f"postgresql+psycopg://{os.getenv('PG_USER')}:{os.getenv('PG_PASSWORD')}"
          f"@{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DB')}")
   engine = create_engine(url)

   with engine.connect() as con:
       print(con.execute(text("SELECT version()")).scalar())

PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [4]:
import os, pathlib
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, event, text

load_dotenv("../.env")
engine_pg = create_engine(
    f"postgresql+psycopg://{os.getenv('PG_USER')}:{os.getenv('PG_PASSWORD')}"
    f"@{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DB')}"
)
engine_sqlite = create_engine("sqlite:///../data/processed/schnee.db")

@event.listens_for(engine_sqlite, "connect")
def fk_an(dbapi_con, _):
    dbapi_con.execute("PRAGMA foreign_keys = ON")   # SQLite prüft Fremdschlüssel nur auf Wunsch

schema = pathlib.Path("../sql/schema.sql").read_text()

P = "../data/processed/"
gem = pd.read_csv(P + "gemeinden.csv")
hot = pd.read_csv(P + "hotellerie_monat.csv")
her = pd.read_csv(P + "herkunft_monat.csv")
wet = pd.read_csv(P + "wetter_monat.csv")
panel = pd.read_csv(P + "panel_winter.csv")
zeit = panel[["jahr", "monat", "winter", "winter_label", "covid", "ostern_im_monat"]].drop_duplicates()

print("zeit:", zeit.shape, "| doppelte Schlüssel:", zeit.duplicated(["jahr", "monat"]).sum())
for n, d in [("hotellerie", hot), ("herkunft", her), ("wetter", wet)]:
    print(n, "| Zeilen ohne Gemeinde:", (~d["bfs_nr"].isin(gem["bfs_nr"])).sum(),
          "| doppelte Schlüssel:", d.duplicated(["bfs_nr", "jahr", "monat"]).sum())

zeit: (52, 6) | doppelte Schlüssel: 0
hotellerie | Zeilen ohne Gemeinde: 0 | doppelte Schlüssel: 0
herkunft | Zeilen ohne Gemeinde: 0 | doppelte Schlüssel: 0
wetter | Zeilen ohne Gemeinde: 0 | doppelte Schlüssel: 0


In [5]:
tabellen = {"gemeinde": gem, "zeit": zeit, "hotellerie": hot, "herkunft": her, "wetter": wet}

def laden(eng):
    with eng.begin() as conn:
        for stmt in schema.split(";"):            # Schema neu anlegen, damit die Zelle wiederholbar ist
            if stmt.strip():
                conn.execute(text(stmt))
        for name, d in tabellen.items():          # Reihenfolge wegen Fremdschlüsseln
            d.to_sql(name, conn, if_exists="append", index=False, chunksize=5000)

laden(engine_sqlite)
print("SQLite fertig")
laden(engine_pg)
print("PostgreSQL fertig")

SQLite fertig
PostgreSQL fertig


In [6]:
zaehlen = text("""
SELECT 'gemeinde' AS tabelle, COUNT(*) AS zeilen FROM gemeinde
UNION ALL SELECT 'zeit', COUNT(*) FROM zeit
UNION ALL SELECT 'hotellerie', COUNT(*) FROM hotellerie
UNION ALL SELECT 'herkunft', COUNT(*) FROM herkunft
UNION ALL SELECT 'wetter', COUNT(*) FROM wetter
UNION ALL SELECT 'v_panel_winter', COUNT(*) FROM v_panel_winter
""")

for name, eng in [("SQLite", engine_sqlite), ("PostgreSQL", engine_pg)]:
    with eng.connect() as conn:
        z = pd.read_sql(zaehlen, conn)
        s = conn.execute(text("SELECT SUM(zimmerauslastung) FROM v_panel_winter")).scalar()
    print(name)
    print(z.to_string(index=False))
    print("Summe Zimmerauslastung (View):", round(s, 4), "| (CSV):", round(panel["zimmerauslastung"].sum(), 4), "\n")

SQLite
       tabelle  zeilen
      gemeinde     186
          zeit      52
    hotellerie   29459
      herkunft   29378
        wetter    7200
v_panel_winter    2255
Summe Zimmerauslastung (View): 120703.0 | (CSV): 120703.0 

PostgreSQL
       tabelle  zeilen
      gemeinde     186
          zeit      52
        wetter    7200
      herkunft   29378
    hotellerie   29459
v_panel_winter    2255
Summe Zimmerauslastung (View): 120703.0 | (CSV): 120703.0 

